### [DENEY 3] — ⭐ EN YÜKSEK BAŞARIMLI MODEL (BEST MODEL)
**Amaç:** Optimize edilmiş en iyi ELA ölçek faktörü (15) ve izole veri bölütlemesi altında, derinleştirilmiş M2 modelinin daha uzun süre eğitildiğinde ulaştığı zirve performansı gözlemlemek.
**Bu Hücrede Değişen Parametreler / Yapılan Ayarlar:**
- **ELA Ölçek Faktörü (`scale`):** 15 (Yüksek kontrastlı anomali haritası aktifleştirildi)
- **Veri Kümesi Bölütleme (Split):** %80 Eğitim, %16 Test, %4 Validasyon (Katmanlı yalıtılmış split)
- **Epoch Sayısı:** 60'a çıkarıldı (M2'deki Batch Normalization sayesinde model overfitting olmadan kararlı şekilde %95.53 doğruluğa ulaştı)
- **Model Yapıları:** M1 baz modeli ve önerilen derin M2 modeli eğitildi.

In [ ]:
import numpy as np
import random
from PIL import Image, ImageChops, ImageEnhance
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

os.environ['KAGGLE_API_TOKEN'] = 'KGAT_0c19a9db6740e7da1d8da96086f765c2'
!pip install kaggle -q
!kaggle datasets download -d sophatvathana/casia-dataset -q
!unzip -q -o casia-dataset.zip -d casia
print("Veri hazır!")

In [ ]:
def image_to_ela(path, quality=90):
    try:
        image = Image.open(path).convert('RGB')
        image.save('resaved.jpg', 'JPEG', quality=quality)
        resaved = Image.open('resaved.jpg')
        ela_image = ImageChops.difference(image, resaved)
        band_values = ela_image.getextrema()
        max_value = max([val[1] for val in band_values])
        if max_value == 0:
            max_value = 1
        scale = 255.0 / max_value * 15  # scale=15
        ela_image = ImageEnhance.Brightness(ela_image).enhance(scale)
        return ela_image
    except Exception as e:
        print(f'Hata: {e}')
        return None

image_size = (150, 150)

def prepare_image(path):
    ela = image_to_ela(path)
    if ela is None:
        return np.zeros((150*150*3,))
    return np.array(ela.resize(image_size)).flatten() / 255.0

In [ ]:
X, Y = [], []

path = 'casia/CASIA2/Au'
for filename in os.listdir(path):
    if filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg'):
        X.append(prepare_image(os.path.join(path, filename)))
        Y.append(1)
        if len(Y) % 1000 == 0:
            print(f'{len(Y)} görüntü işlendi')

print(f"Orijinal: {len(X)}")

path = 'casia/CASIA2/Tp'
for filename in os.listdir(path):
    if filename.lower().endswith('.jpg') or filename.lower().endswith('.jpeg'):
        X.append(prepare_image(os.path.join(path, filename)))
        Y.append(0)
        if len(Y) % 1000 == 0:
            print(f'{len(Y)} görüntü işlendi')

print(f"Toplam: {len(X)}")

for i in range(10):
    X, Y = shuffle(X, Y, random_state=i)

X = np.array(X).reshape(-1, 150, 150, 3)
Y = to_categorical(Y, 2)
print(f"X shape: {X.shape}, Y shape: {Y.shape}")


In [ ]:
X_train, X_rest, Y_train, Y_rest = train_test_split(
    X, Y, test_size=0.2, random_state=42)
del X, Y

X_test, X_val, Y_test, Y_val = train_test_split(
    X_rest, Y_rest, test_size=0.2, random_state=42)
del X_rest, Y_rest

print(f"Train: {len(X_train)}, Test: {len(X_test)}, Val: {len(X_val)}")

In [ ]:
model1 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(150, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='sigmoid')
])

model1.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(), Recall()]
)

model1.summary()


In [ ]:
#6.hücre
history1 = model1.fit(
    X_train, Y_train,
    epochs=40,
    batch_size=8,
    validation_data=(X_val, Y_val),
    verbose=1
)
print("Model 1 tamamlandı!")


In [ ]:
model2 = models.Sequential([
    layers.Input(shape=(150, 150, 3)),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (5,5), padding='valid', activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Conv2D(64, (5,5), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (5,5), padding='same', activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='sigmoid')
])

model2.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', Precision(), Recall()]
)

model2.summary()

In [ ]:
history2 = model2.fit(
    X_train, Y_train,
    epochs=60,
    batch_size=8,
    validation_data=(X_val, Y_val),
    verbose=1
)
print("Model 2 tamamlandı!")


In [ ]:
# Model 1
loss1, acc1, prec1, rec1 = model1.evaluate(X_test, Y_test)
y_pred1 = model1.predict(X_test)
y_pred1_labels = np.argmax(y_pred1, axis=1)
y_true_labels = np.argmax(Y_test, axis=1)

print("\n=== Model 1 - Reproduce ===")
print(f"Accuracy: {acc1:.4f}, Precision: {prec1:.4f}, Recall: {rec1:.4f}")
print(classification_report(y_true_labels, y_pred1_labels,
      target_names=['Sahte', 'Orijinal']))

# Model 2
loss2, acc2, prec2, rec2 = model2.evaluate(X_test, Y_test)
y_pred2 = model2.predict(X_test)
y_pred2_labels = np.argmax(y_pred2, axis=1)

print("\n=== Model 2 - İyileştirme ===")
print(f"Accuracy: {acc2:.4f}, Precision: {prec2:.4f}, Recall: {rec2:.4f}")
print(classification_report(y_true_labels, y_pred2_labels,
      target_names=['Sahte', 'Orijinal']))

In [ ]:
print("=" * 60)
print("SONUÇ KARŞILAŞTIRMASI")
print("=" * 60)
print(f"Makale (scale=?)     : Acc=%94.14, Prec=%94.1, Rec=%94.07")
print(f"Model 1 (scale=1)    : Acc=%93.62, Prec=%93.68, Rec=%93.68")
print(f"Model 2 (scale=1)    : Acc=%93.22, Prec=%93.28, Rec=%93.09")
print(f"Model 1 (scale=15)   : Acc={acc1*100:.2f}, Prec={prec1*100:.2f}, Rec={rec1*100:.2f}")
print(f"Model 2 (scale=15)   : Acc={acc2*100:.2f}, Prec={prec2*100:.2f}, Rec={rec2*100:.2f}")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(history1.history['accuracy'], color='b', label='Train')
axes[0,0].plot(history1.history['val_accuracy'], color='r', label='Val')
axes[0,0].set_title('Model 1 - Accuracy'); axes[0,0].legend()

axes[0,1].plot(history1.history['loss'], color='b', label='Train')
axes[0,1].plot(history1.history['val_loss'], color='r', label='Val')
axes[0,1].set_title('Model 1 - Loss'); axes[0,1].legend()

axes[1,0].plot(history2.history['accuracy'], color='b', label='Train')
axes[1,0].plot(history2.history['val_accuracy'], color='r', label='Val')
axes[1,0].set_title('Model 2 - Accuracy'); axes[1,0].legend()

axes[1,1].plot(history2.history['loss'], color='b', label='Train')
axes[1,1].plot(history2.history['val_loss'], color='r', label='Val')
axes[1,1].set_title('Model 2 - Loss'); axes[1,1].legend()

plt.tight_layout()
plt.savefig('/kaggle/working/tum_grafikler_scale15.png', dpi=150)
plt.show()
print("Her şey tamamlandı!")